# CalCourse Exploratory Data Analysis

This notebook explores the Fall 2026 Berkeley course and class datasets used in CalCourse. 

Goals are to: 
- understand the structure of the data
- identify missing or inconsistent fields 
- define the undergraduate recommendation pool
- inspect prerequisite and course metadata before modeling

## 1. Load Data

In [ ]:
import pandas as pd

courses = pd.read_csv("../data/processed/courses_fall_2026.csv")
classes = pd.read_csv("../data/processed/classes_fall_2026.csv")

## 2. Dataset Overview
Inspecting dataset dimensions, schema, and sample record. 

In [ ]:
print(courses.shape)
print(classes.shape)
print(courses.head())
print(classes.head())

In [ ]:
courses.info()
classes.info()

### Initial observations 
- The course dataset contains 4,287 unique courses. 
- The class dataset contains 15,573 Fall 2026 class records. 
- Course-level and class-level data are stored separately because one course may correspond to multiple class offerings.

## 3. Missing Data
Evaluating which fields are complete enough to use and which may require exclusion or special handling. 

In [ ]:
courses.isna().sum().sort_values(ascending=False)
# classes.isna().sum().sort_values(ascending=False)

In [ ]:
courses.loc[
        courses["requirements"].notna(),
        ["subject", "course_number", "title", "academic_career"]
].head(20)

In [ ]:
courses.loc[
    courses["description"].isna(),
    ["subject", "course_number", "title", "academic_career"]
].head(20)

### Missing data observations 
- 'department_nicknames' and 'typically_offered' are completely null and will likely be excluded. 
- 'requirements' is missing for many courses, but they may just indicate the absence of explicit prerequisites rather than a data-quality issue. 
- Course titles, identifiers, subjects, academic career, and department fields are complete.
- Missing descriptions are relatively uncommon and appear to be concentrated in certain research, seminar, or special-topic courses. 

## 4. Course Population 
Examining the size and composition of the course catalog to define the recommendation pool.

In [ ]:
courses["academic_career"].value_counts()

In [ ]:
courses["subject"].nunique()

In [ ]:
courses["department"].value_counts().head(20)

In [ ]:
courses["course_id"].nunique(), len(courses)

In [ ]:
# CalCourse V1 will filter for undergrad

ug = courses[courses["academic_career"] == "UGRD"]
ug.shape

In [ ]:
ug["course_number"].sample(50, random_state=42)

### Course Population Observations
- The dataset contains 4,287 unique Fall 2026 courses. 
- 2,306 courses are undergraduate and will form the initial CalCourse recommendation pool. 
- The catalog spans 212 subjects which provides broad coverage across departments. 
- Course numbers contain non-numeric prefixes and suffixes such as 'C', 'H', and 'AC', so they must be treated as strings rather than numeric values. 

## 5. Prerequisite Structure
Prereq information is stored as semi-structured text. This section looks at how prerequisites are written and how they can be handled in the recommendation system. 

In [ ]:
# Prerequisite inspection

ug.loc[
    ug["requirements"].notna(),
    ["subject", "course_number", "title", "requirements"]
].sample(
    20, 
    random_state=42
)